<b>Aktywność 7</b><br>
Niezbędne biblioteki: geopandas, pandas, pyproj

<b>Ćwiczenie 1.</b> Wczytaj dane z pliku ``shopping_centres.txt`` do obiektu typu DataFrame o nazwie shopping_centres.

In [162]:
import pandas as pd

### your code ###
shopping_centres = pd.read_csv("shopping_centres.txt", sep=';')

Zweryfikuj poprawność kodu.

In [163]:
assert isinstance(shopping_centres, pd.DataFrame)
for column in ('id', 'name', 'addr'):
    assert column in shopping_centres.columns

<b>Ćwiczenie 2.</b> Przeprowadź geokodowanie adresów z wykorzystaniem zmiennej geocoded_addresses i geokodera Nominatim.

In [164]:
from geopandas.tools import geocode
### your code ###
geocoded_addresses = geocode(shopping_centres['addr'], provider='nominatim', user_agent='Analiza7')

<b>Ćwiczenie 3.</b> Złącz wynik geokodowania z danymi wejściowymi i zapisz je w obiekcie typu GeoDataFrame o nazwie shopping_centres.

In [165]:
import geopandas as gpd

### your code ###
shopping_centres = gpd.GeoDataFrame(shopping_centres, geometry=geocoded_addresses['geometry'])

Zweryfikuj poprawność kodu.

In [166]:
assert isinstance(shopping_centres, gpd.GeoDataFrame)
for column in ('id', 'name', 'addr', 'geometry'):
    assert column in shopping_centres.columns

<b>Ćwiczenie 4.</b> Przeprowadź projekcję danych na EPSG:3879.

In [167]:
### your code ###
shopping_centres = shopping_centres.to_crs(epsg=3879)

Zweryfikuj poprawność kodu.

In [168]:
import pyproj

assert shopping_centres.crs == pyproj.CRS('EPSG:3879')

<b>Ćwiczenie 5.</b> Zapisz plik w formacie ShapeFile pod nazwą ``shopping_centres.shp``.

In [169]:
### your code ###
output_file = "shopping_centres.shp"
shopping_centres.to_file(output_file, driver='ESRI Shapefile')

<b>Ćwiczenie 6.</b> Oblicz bufor o promieniu 1.5 km od każdego centrum handlowego. Nadpisz wynik w kolumnie geometry.

In [170]:
### your code ###
shopping_centres['geometry'] = shopping_centres['geometry'].buffer(1500)

Zweryfikuj poprawność kodu.

In [171]:
assert shopping_centres.geometry.geom_type.unique() == ['Polygon']

<b>Ćwiczenie 7.</b> Pobierz dane dotyczące populacji z wykorzystaniem zmiennej population_grid.

In [172]:
population_grid = gpd.read_file(
    (
        'https://kartta.hsy.fi/geoserver/wfs'
        '?service=wfs'
        '&version=2.0.0'
        '&request=GetFeature'
        '&typeName=asuminen_ja_maankaytto:Vaestotietoruudukko_2020'
        '&srsName=EPSG:3879'
    ),
)
#population_grid.set_crs(epsg='3879', inplace=True, allow_override=True)
population_grid.crs = "EPSG:3879"

Zweryfikuj poprawność kodu.

In [173]:
assert isinstance(population_grid, gpd.GeoDataFrame)
assert population_grid.crs == pyproj.CRS('EPSG:3879')

<b>Ćwiczenie 8.</b> Wykonaj odpowiednie złączenie przestrzenne pomiędzy obiektami shopping_centres i population_grid z wykorzystaniem zmiennej populations. Następnie usuń wszystkie kolumny poza name i asukkaita (population).

In [174]:
### your code ###
populations = gpd.sjoin(population_grid, shopping_centres, how='inner', predicate='within')
#populations.head()
populations = populations[['name','asukkaita']]
populations.head()

,name,asukkaita
1092,Iso-omena,47
1146,Iso-omena,145
1147,Iso-omena,128
1148,Iso-omena,81
1149,Iso-omena,20


<b>Ćwiczenie 9.</b> Dla każdego centrum handlowego oblicz liczbę mieszkańców żyjących w promieniu 1.5 km od niego. Rezultat nadpisz w zmiennej populations.

In [176]:
### your code ###
populations.groupby(['name'])['asukkaita'].agg('sum')

name
Forum        57242
Iso-omena    27612
Itis         20098
Jumbo        10956
REDI         28752
Sello        23429
Tripla       23868
Name: asukkaita, dtype: int64